# Stage 4 — MATLAB ↔ Python GPU RIS Response

Official model:

\[
\phi\in\{45^\circ,135^\circ\}
\]

\[
\beta(\phi)
=
(1-0.8)
\left(
\frac{\sin(\phi-0.43\pi)+1}{2}
\right)^{1.6}
+0.8
\]

\[
\gamma=\beta e^{j\phi}.
\]

Bu notebook:

\[
z\rightarrow\phi\rightarrow\beta\rightarrow\gamma
\]

zincirini MATLAB ile karşılaştırır.

Test edilen RIS boyutları:

\[
64,\ 128,\ 256,\ 512.
\]

Her boyutta 32 pattern kullanılır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, zipfile, shutil, json
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

MODULE = ROOT / 'ris_gpu_ris_response_stage4.py'
if not MODULE.exists():
    MODULE = Path('/content/ris_gpu_ris_response_stage4.py')

assert MODULE.exists(), (
    "ris_gpu_ris_response_stage4.py dosyasını "
    "Drive RIS root'a veya /content altına yükle."
)

sys.path.insert(0,str(MODULE.parent))

from ris_gpu_ris_response_stage4 import (
    compare_stage4_matlab_case,
    benchmark_ris_response,
    generate_ris_response_from_z,
)

print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :",torch.cuda.get_device_name(0))
print("Loaded:",MODULE)

## MATLAB golden suite

MATLAB'da:

```matlab
export_stage4_parity_suite
```

çalıştır.

Oluşan:

```text
stage4_ris_golden.zip
```

dosyasını Colab `/content` altına yükle.

In [ ]:
ZIP = Path('/content/stage4_ris_golden.zip')
assert ZIP.exists(), "stage4_ris_golden.zip dosyasını /content altına yükle."

EXTRACT = Path('/content/stage4_ris_extract')
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(ZIP,'r') as zf:
    zf.extractall(EXTRACT)

manifests = list(EXTRACT.rglob('manifest.csv'))
assert len(manifests) == 1, manifests

SUITE = manifests[0].parent
manifest = pd.read_csv(manifests[0])

display(manifest)

assert set(manifest['nRIS'].astype(int)) == {64,128,256,512}
assert (manifest['nPatterns'].astype(int) == 32).all()

print("PASS: expected nRIS coverage")

In [ ]:
# FLOAT64 / COMPLEX128 PARITY

device = 'cuda' if torch.cuda.is_available() else 'cpu'

rows = []

for _,r in manifest.iterrows():
    p = SUITE / str(r['file'])
    m = compare_stage4_matlab_case(
        str(p),
        device=device,
        parity=True,
    )
    rows.append({
        'nRIS':int(r['nRIS']),
        **m
    })

df64 = pd.DataFrame(rows)
display(df64)

worst64 = max(
    df64['phi_relFro'].max(),
    df64['beta_relFro'].max(),
    df64['gamma_relFro'].max(),
)

print("Worst double relative error:",worst64)

assert worst64 < 1e-12, (
    f"Stage 4 double parity failed: {worst64:.3e}"
)

print("PASS: Stage 4 double MATLAB parity")

In [ ]:
# FLOAT32 / COMPLEX64 PRODUCTION ERROR

rows = []

for _,r in manifest.iterrows():
    p = SUITE / str(r['file'])
    m = compare_stage4_matlab_case(
        str(p),
        device=device,
        parity=False,
    )
    rows.append({
        'nRIS':int(r['nRIS']),
        **m
    })

df32 = pd.DataFrame(rows)
display(df32)

worst32 = max(
    df32['phi_relFro'].max(),
    df32['beta_relFro'].max(),
    df32['gamma_relFro'].max(),
)

print("Worst float32 relative error:",worst32)

assert worst32 < 1e-5, (
    f"Stage 4 float32 sanity failed: {worst32:.3e}"
)

print("PASS: Stage 4 float32 production sanity")

In [ ]:
# GPU batch throughput
if torch.cuda.is_available():
    bench = benchmark_ris_response(
        n_candidates=4096,
        n_ris=512,
        repeats=20,
        device='cuda',
    )
    print(json.dumps(bench,indent=2))

## Stage 4 kabul kriteri

Stage 4 tamamlanmış sayılırsa:

\[
\max e_{\rm double}<10^{-12}
\]

ve

\[
\max e_{\rm fp32}<10^{-5}.
\]

Sonraki aşama Type-I rank-1 codebook / precoder parity olacaktır.